In [4]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
log = logging.getLogger("weather_pipeline")

log.info("Bắt đầu thử logging")
log.warning("Đây là cảnh báo")
log.error("Đây là lỗi")

2026-10-02 11:37:07,877 | INFO | Bắt đầu thử logging
2026-10-02 11:37:07,879 | WARNING | Đây là cảnh báo
2026-10-02 11:37:07,881 | ERROR | Đây là lỗi


In [5]:
import time
import requests

def get_json(url, params=None, headers=None, retries=4, timeout=10):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as e:
            log.warning(f"Lần {attempt}/{retries}: lỗi mạng ({type(e).__name__})")
        else:
            if r.status_code < 400:
                return r.json()
            if r.status_code != 429 and r.status_code < 500:
                raise RuntimeError(f"Lỗi {r.status_code}, không retry: {r.text[:200]}")
            log.warning(f"Lần {attempt}/{retries}: server trả {r.status_code}")
        if attempt < retries:
            wait = 2 ** attempt
            log.info(f"Chờ {wait} giây rồi thử lại")
            time.sleep(wait)
    raise RuntimeError(f"Hết {retries} lần thử: {url}")

In [6]:
URL = "https://api.open-meteo.com/v1/forecast"

CITIES = {
    "Ha Noi": (21.03, 105.85),
    "TP HCM": (10.82, 106.63),
    "Da Nang": (16.05, 108.20),
}

def extract_weather(cities, days=7):
    raw = {}
    for city, (lat, lon) in cities.items():
        params = {
            "latitude": lat,
            "longitude": lon,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
            "timezone": "Asia/Ho_Chi_Minh",
            "forecast_days": days,
        }
        log.info(f"Lấy dữ liệu: {city}")
        raw[city] = get_json(URL, params=params)
        time.sleep(1)
    log.info(f"Đã lấy xong {len(raw)} thành phố")
    return raw

raw = extract_weather(CITIES)
print(list(raw.keys()))
print(raw["Ha Noi"]["daily"]["time"])

2026-10-02 11:45:13,510 | INFO | Lấy dữ liệu: Ha Noi
2026-10-02 11:45:16,949 | INFO | Lấy dữ liệu: TP HCM
2026-10-02 11:45:19,010 | INFO | Lấy dữ liệu: Da Nang
2026-10-02 11:45:21,185 | INFO | Đã lấy xong 3 thành phố


['Ha Noi', 'TP HCM', 'Da Nang']
['2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08']


In [7]:
import pandas as pd

def transform_weather(raw):
    frames = []
    for city, data in raw.items():
        df_city = pd.DataFrame(data["daily"])
        df_city["city"] = city
        frames.append(df_city)
    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"]).dt.date
    df["ingested_at"] = pd.Timestamp.now(tz="UTC")
    log.info(f"Transform xong: {df.shape[0]} dòng, {df.shape[1]} cột")
    return df

df = transform_weather(raw)
print(df.dtypes)
df.head(3)

2026-10-02 12:21:33,407 | INFO | Transform xong: 21 dòng, 6 cột


time                               object
temperature_2m_max                float64
temperature_2m_min                float64
precipitation_sum                 float64
city                                  str
ingested_at           datetime64[us, UTC]
dtype: object


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,city,ingested_at
0,2026-10-02,33.9,27.1,2.3,Ha Noi,2026-10-02 05:21:33.405668+00:00
1,2026-10-03,33.5,26.5,1.2,Ha Noi,2026-10-02 05:21:33.405668+00:00
2,2026-10-04,29.2,24.2,21.1,Ha Noi,2026-10-02 05:21:33.405668+00:00


In [8]:
def validate_weather(df, expected_cities=3, days=7):
    problems = []

    if len(df) != expected_cities * days:
        problems.append(f"Số dòng là {len(df)}, mong đợi {expected_cities * days}")

    nulls = df.isna().sum().sum()
    if nulls > 0:
        problems.append(f"Có {nulls} giá trị thiếu")

    if df.duplicated(subset=["city", "time"]).any():
        problems.append("Có dòng trùng (city, time)")

    if (df["temperature_2m_max"] < df["temperature_2m_min"]).any():
        problems.append("Có dòng nhiệt độ max nhỏ hơn min")

    if (df["precipitation_sum"] < 0).any():
        problems.append("Có lượng mưa âm")

    if problems:
        for p in problems:
            log.error(p)
        raise ValueError("Dữ liệu không đạt kiểm tra: " + "; ".join(problems))

    log.info("Kiểm tra dữ liệu: đạt")

validate_weather(df)

2026-10-02 12:31:09,575 | INFO | Kiểm tra dữ liệu: đạt


In [9]:
df_bad = df.copy()
df_bad = df_bad.drop(index=0)
df_bad.loc[1, "precipitation_sum"] = None
df_bad.loc[2, "temperature_2m_max"] = 0

try:
    validate_weather(df_bad)
except ValueError as e:
    print("Pipeline đã dừng:", e)

print("df gốc vẫn còn", len(df), "dòng")

2026-10-02 12:39:47,616 | ERROR | Số dòng là 20, mong đợi 21
2026-10-02 12:39:47,619 | ERROR | Có 1 giá trị thiếu
2026-10-02 12:39:47,623 | ERROR | Có dòng nhiệt độ max nhỏ hơn min


Pipeline đã dừng: Dữ liệu không đạt kiểm tra: Số dòng là 20, mong đợi 21; Có 1 giá trị thiếu; Có dòng nhiệt độ max nhỏ hơn min
df gốc vẫn còn 21 dòng


In [10]:
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
TABLE_ID = f"{PROJECT}.raw.weather_daily"
client = bigquery.Client(project=PROJECT, location="US")

SCHEMA = [
    bigquery.SchemaField("time", "DATE"),
    bigquery.SchemaField("temperature_2m_max", "FLOAT"),
    bigquery.SchemaField("temperature_2m_min", "FLOAT"),
    bigquery.SchemaField("precipitation_sum", "FLOAT"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("ingested_at", "TIMESTAMP"),
]

def load_weather(df, table_id=TABLE_ID):
    job_config = bigquery.LoadJobConfig(
        schema=SCHEMA,
        write_disposition="WRITE_TRUNCATE",
    )
    log.info(f"Nạp {len(df)} dòng vào {table_id}")
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    if job.output_rows != len(df):
        raise RuntimeError(f"Nạp {job.output_rows} dòng, mong đợi {len(df)}")
    log.info(f"Nạp xong: {job.output_rows} dòng")
    return job.output_rows

load_weather(df)

2026-10-02 12:48:23,463 | INFO | Nạp 21 dòng vào bq-learning-510104.raw.weather_daily
2026-10-02 12:48:29,012 | INFO | Nạp xong: 21 dòng


21

In [11]:
def check(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Sẽ quét khoảng {job.total_bytes_processed / 1024**2:.4f} MB")

def run(sql, max_mb=100):
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=max_mb * 1024**2)
    return client.query(sql, job_config=cfg).to_dataframe()

sql = f"""
SELECT city,
       COUNT(*) AS days,
       MIN(time) AS first_day,
       MAX(time) AS last_day,
       MAX(ingested_at) AS last_loaded
FROM `{TABLE_ID}`
GROUP BY city
"""
check(sql)
run(sql)

Sẽ quét khoảng 0.0005 MB


,city,days,first_day,last_day,last_loaded
0,Ha Noi,7,2026-10-02,2026-10-08,2026-10-02 05:21:33.405668+00:00
1,TP HCM,7,2026-10-02,2026-10-08,2026-10-02 05:21:33.405668+00:00
2,Da Nang,7,2026-10-02,2026-10-08,2026-10-02 05:21:33.405668+00:00
